# 6.10 — Second-Order Methods & K-FAC

Second-order optimization asks not only "which way is downhill?" but also "how curved is the hill in each direction?" K-FAC (Kronecker-Factored Approximate Curvature) makes that idea practical for neural-network layers: instead of inverting one enormous curvature matrix, it approximates the layer curvature with two small covariance factors and applies the update \(\Delta W \approx -\eta A^{-1} G S^{-1}\).

## 📖 Concept walkthrough — build each idea from scratch

Before the compact worked examples, we build second-order optimization and K-FAC one idea at a time. Run each cell in order and inspect the numbers: the goal is to see why raw gradients can be poorly scaled, how curvature rescales them, and why Kronecker structure turns an impossible inverse into two small ones. The walkthrough uses `_w` variables so it does not collide with later examples.

In [ ]:
import numpy as np  # arrays and linear algebra for every scratch calculation.
import matplotlib.pyplot as plt  # visual debugging for losses, steps, and matrices.
np.random.seed(0)  # reproducibility for minibatches and toy factors.

### 1. Raw gradient descent can be fooled by curvature

Start with the smallest possible optimization problem: a quadratic bowl \(L(w)=\tfrac12 h(w-w^*)^2\). Its gradient is \(h(w-w^*)\), so the same distance from optimum creates a much larger gradient when curvature \(h\) is large. A single global learning rate must be tiny enough for the steep direction, which makes shallow directions crawl.

In [ ]:
w_grid_w = np.linspace(-1.0, 4.0, 120)  # possible parameter values to visualize.
w_star_w = 1.5  # true minimizer of the toy loss.
h_flat_w, h_steep_w = 1.0, 12.0  # two curvatures for the same optimum.
loss_flat_w = 0.5 * h_flat_w * (w_grid_w - w_star_w) ** 2  # shallow quadratic.
loss_steep_w = 0.5 * h_steep_w * (w_grid_w - w_star_w) ** 2  # steep quadratic.
print("flat gradient at w=3:", h_flat_w * (3.0 - w_star_w))
print("steep gradient at w=3:", h_steep_w * (3.0 - w_star_w))
assert h_steep_w * (3.0 - w_star_w) == 18.0

▶ What you'll see: the steep bowl produces a gradient of 18 at the same location where the flat bowl produces 1.5.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(w_grid_w, loss_flat_w, label="h=1 shallow")
plt.plot(w_grid_w, loss_steep_w, label="h=12 steep")
plt.axvline(w_star_w, color="black", linestyle="--", linewidth=1)
plt.scatter([3.0], [0.5 * h_steep_w * (3.0 - w_star_w) ** 2], color="red")
plt.title("1: same optimum, different curvature")
plt.xlabel("parameter w"); plt.ylabel("loss")
plt.legend(); plt.show()

▶ What you'll see: both losses minimize at 1.5, but the steep one rises much faster away from that point.

*Why it's done this way:* a gradient mixes two facts: direction to improve and local scale. The derivative \(h(w-w^*)\) is large either because we are far away or because the bowl is sharp. A second-order method separates those facts by dividing by curvature, so the update reflects distance-to-optimum rather than raw slope size.

### 2. Newton's step divides by curvature

For the same quadratic, the Hessian is just \(h\). Newton's update is \(w \leftarrow w - g/h\), so from any starting point it jumps exactly to \(w^*\) on a perfect quadratic. The lesson's scalar update \(2.000 - 0.050\cdot1.650=1.917\) is a first-order nudge; Newton asks whether the gradient should be shrunken or enlarged by the local curvature before nudging.

In [ ]:
w0_w = 3.0  # start to the right of the optimum.
g_w = h_steep_w * (w0_w - w_star_w)  # gradient in the steep bowl.
newton_step_w = g_w / h_steep_w  # curvature-corrected step length.
w_newton_w = w0_w - newton_step_w  # Newton update.
w_gd_w = w0_w - 0.05 * g_w  # ordinary gradient descent with a small learning rate.
print("gradient:", round(g_w, 3), "Newton step:", round(newton_step_w, 3))
print("GD w:", round(w_gd_w, 3), "Newton w:", round(w_newton_w, 3))
assert round(w_gd_w, 3) == 2.1
assert round(w_newton_w, 3) == 1.5

▶ What you'll see: gradient descent moves from 3.0 to 2.1, while Newton lands on 1.5 in one step.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(w_grid_w, loss_steep_w, color="purple")
plt.scatter([w0_w, w_gd_w, w_newton_w], [0.5*h_steep_w*(w0_w-w_star_w)**2,
                                         0.5*h_steep_w*(w_gd_w-w_star_w)**2,
                                         0.5*h_steep_w*(w_newton_w-w_star_w)**2],
            color=["red", "orange", "green"])
plt.title("2: curvature-corrected step")
plt.xlabel("w"); plt.ylabel("loss"); plt.show()

▶ What you'll see: the green Newton point is at the bottom, while the orange gradient-descent point is only partway there.

*Why it's done this way:* on a quadratic, the second-order Taylor model is exact: \(L(w+\Delta)\approx L(w)+g\Delta+\tfrac12 h\Delta^2\). Minimizing that approximation gives \(g+h\Delta=0\), hence \(\Delta=-g/h\). K-FAC keeps this same "divide by curvature" logic but replaces the impossible full Hessian with layerwise covariance factors.

### 3. A layer gradient is an outer product

For one linear layer with activations \(a\), pre-activations \(s=aW\), and backpropagated derivatives \(\delta=\partial L/\partial s\), the weight gradient is \(G=a^\top\delta\) for a single example and an average of outer products for a minibatch. This shape fact is why K-FAC can separate curvature into an activation factor and a gradient factor.

In [ ]:
a_w = np.array([[1.5, -0.5]])  # one input activation row, matching the lesson's scratch pass.
W_w = np.array([[1.0], [-0.1]])  # two weights into one unit.
b0_w = 0.7  # bias from the lesson content.
z_w = float(a_w @ W_w + b0_w)  # affine signal.
relu_w = max(0.0, z_w)  # gated signal.
print("affine signal:", round(z_w, 3), "ReLU signal:", round(relu_w, 3))
assert round(z_w, 3) == 2.25

▶ What you'll see: the two-input affine computation gives 2.25, and ReLU leaves it unchanged because it is positive.

In [ ]:
delta_w = np.array([[1.65]])  # derivative arriving from the loss for this one unit.
G_one_w = a_w.T @ delta_w  # outer product: input coordinate by output derivative.
print("single-example gradient:\n", np.round(G_one_w, 3))
assert np.allclose(G_one_w.ravel(), [2.475, -0.825])

▶ What you'll see: the first weight gets a positive gradient and the second gets a negative gradient.

In [ ]:
a_batch_w = np.array([[1.5, -0.5], [0.5, 1.0], [1.0, 0.0], [-1.0, 0.5]])
delta_batch_w = np.array([[1.65, -0.20], [0.50, 0.30], [1.00, -0.40], [-0.30, 0.20]])
G_batch_w = a_batch_w.T @ delta_batch_w / a_batch_w.shape[0]
print("batch gradient shape:", G_batch_w.shape)
print(np.round(G_batch_w, 3))
assert G_batch_w.shape == (2, 2)

▶ What you'll see: a 2×2 gradient matrix, one entry for each input-output weight connection.

*Why it's done this way:* backprop gives a local output-side error \(\delta\), but a weight only matters in proportion to the activation that flowed through it. Multiplying activation by derivative creates exactly the right local credit assignment. K-FAC's approximation starts from the observation that these two ingredients often have useful second-moment structure.

### 4. Curvature for a layer is enormous if we flatten every weight

If a layer has \(d_{in}\) inputs and \(d_{out}\) outputs, the flattened weight vector has \(d_{in}d_{out}\) entries. A full curvature matrix over those entries has \((d_{in}d_{out})^2\) numbers, and inverting it is the bottleneck. K-FAC avoids that by approximating the layer curvature as \(S\otimes A\), where \(A=E[a^\top a]\) and \(S=E[\delta^\top\delta]\).

In [ ]:
A_w = a_batch_w.T @ a_batch_w / a_batch_w.shape[0]  # activation covariance factor.
S_w = delta_batch_w.T @ delta_batch_w / delta_batch_w.shape[0]  # derivative covariance factor.
print("A factor:\n", np.round(A_w, 3))
print("S factor:\n", np.round(S_w, 3))
assert A_w.shape == (2, 2) and S_w.shape == (2, 2)

▶ What you'll see: two small 2×2 matrices summarizing input scale and output-gradient scale.

In [ ]:
F_kron_w = np.kron(S_w, A_w)  # full flattened curvature approximation for this tiny layer.
print("Kronecker curvature shape:", F_kron_w.shape)
print(np.round(F_kron_w, 3))
assert F_kron_w.shape == (4, 4)

▶ What you'll see: the two 2×2 factors expand into a 4×4 curvature matrix for the four flattened weights.

In [ ]:
din_w, dout_w = 1024, 1024
full_numbers_w = (din_w * dout_w) ** 2
factor_numbers_w = din_w ** 2 + dout_w ** 2
print("full curvature entries:", f"{full_numbers_w:,}")
print("K-FAC factor entries:", f"{factor_numbers_w:,}")
print("compression ratio:", int(full_numbers_w / factor_numbers_w))
assert int(full_numbers_w / factor_numbers_w) == 524288

▶ What you'll see: storing two factors is over 500,000× smaller than storing the full layer curvature at this size.

*Why it's done this way:* the identity \((S\otimes A)^{-1}=S^{-1}\otimes A^{-1}\) means we can act like we used a large curvature matrix while only inverting two small matrices. The approximation is a modeling choice: it assumes activation correlations and output-gradient correlations capture the most important curvature interactions separately.

### 5. The K-FAC update is two-sided preconditioning

K-FAC applies the inverse factors directly to the matrix-shaped gradient: \(\Delta W=-\eta A^{-1}GS^{-1}\). Left-multiplying by \(A^{-1}\) corrects input/activation scale; right-multiplying by \(S^{-1}\) corrects output-gradient scale. This is the concrete core formula from the lesson content.

In [ ]:
eta_w = 0.05  # small learning rate from the lesson content.
damp_w = 0.1  # damping keeps inverses stable.
A_damped_w = A_w + damp_w * np.eye(A_w.shape[0])
S_damped_w = S_w + damp_w * np.eye(S_w.shape[0])
A_inv_w = np.linalg.inv(A_damped_w)
S_inv_w = np.linalg.inv(S_damped_w)
Delta_kfac_w = -eta_w * A_inv_w @ G_batch_w @ S_inv_w
Delta_sgd_w = -eta_w * G_batch_w
print("SGD step:\n", np.round(Delta_sgd_w, 3))
print("K-FAC step:\n", np.round(Delta_kfac_w, 3))
assert Delta_kfac_w.shape == G_batch_w.shape

▶ What you'll see: the K-FAC step is not just a smaller copy of the gradient; it rotates and rescales entries by curvature.

In [ ]:
norm_sgd_w = float(np.linalg.norm(Delta_sgd_w))
norm_kfac_w = float(np.linalg.norm(Delta_kfac_w))
print("||SGD step||:", round(norm_sgd_w, 4))
print("||K-FAC step||:", round(norm_kfac_w, 4))
assert round(norm_sgd_w, 4) == 0.0519
plt.figure(figsize=(4.6, 3))
plt.bar(["SGD", "K-FAC"], [norm_sgd_w, norm_kfac_w], color=["gray", "seagreen"])
plt.title("5: curvature changes step size")
plt.ylabel("Frobenius norm of update"); plt.show()

▶ What you'll see: the two update norms differ because K-FAC measures distance in a curvature-aware geometry.

*Why it's done this way:* ordinary SGD treats every weight coordinate as equally scaled. But if one input feature has high variance or one output derivative is noisy, equal coordinate steps are not equal functional changes. The preconditioner whitens those two sides so the step is closer to a natural-gradient step in the layer's local coordinates.

### 6. Damping and minibatches keep the approximation usable

Covariance factors estimated from a minibatch can be singular or noisy. Damping adds \(\lambda I\) before inversion, making every eigenvalue at least \(\lambda\). That sacrifices some pure Newton aggressiveness for stability — the same practical theme as the lesson's scale, softmax, and memory bookkeeping.

In [ ]:
skinny_batch_w = np.array([[1.0, 2.0], [2.0, 4.0]])  # second column is exactly twice the first.
A_singular_w = skinny_batch_w.T @ skinny_batch_w / skinny_batch_w.shape[0]
eigs_raw_w = np.linalg.eigvalsh(A_singular_w)
eigs_damped_w = np.linalg.eigvalsh(A_singular_w + 0.1 * np.eye(2))
print("raw eigenvalues:", np.round(eigs_raw_w, 6))
print("damped eigenvalues:", np.round(eigs_damped_w, 6))
assert round(float(eigs_raw_w[0]), 6) == 0.0

▶ What you'll see: the raw factor has a zero eigenvalue, while damping lifts it to 0.1.

In [ ]:
cond_raw_w = np.inf if eigs_raw_w[0] == 0 else eigs_raw_w[-1] / eigs_raw_w[0]
cond_damped_w = eigs_damped_w[-1] / eigs_damped_w[0]
print("raw condition:", cond_raw_w)
print("damped condition:", round(float(cond_damped_w), 3))
assert round(float(cond_damped_w), 3) == 126.0
plt.figure(figsize=(4.6, 3))
plt.bar(["raw small eig", "damped small eig"], [eigs_raw_w[0], eigs_damped_w[0]], color=["crimson", "seagreen"])
plt.title("6: damping prevents divide-by-zero")
plt.ylabel("smallest eigenvalue"); plt.show()

▶ What you'll see: damping turns an impossible inverse into a stable, finite preconditioner.

*Why it's done this way:* preconditioning divides by estimated curvature. Dividing by zero or by a noisy tiny number would explode the update. Damping says "trust the curvature estimate, but not infinitely," which is why practical second-order methods look like careful engineering rather than magic.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Curvature scales gradients

The same distance from the optimum creates a much larger gradient in a steeper quadratic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t1_grid = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0])  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
print("parameter grid:", t1_grid.tolist())  # -> [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
t1_star = 1.0  # -> 1.0
print("optimum:", t1_star)  # -> 1.0
t1_h_flat = 1.0  # -> 1.0
print("flat curvature:", t1_h_flat)  # -> 1.0
t1_h_steep = 10.0  # -> 10.0
print("steep curvature:", t1_h_steep)  # -> 10.0
t1_loss_flat = 0.5 * t1_h_flat * (t1_grid - t1_star) ** 2  # -> [4.5, 2.0, 0.5, 0.0, 0.5, 2.0]
print("flat losses:", np.round(t1_loss_flat, 3).tolist())  # -> [4.5, 2.0, 0.5, 0.0, 0.5, 2.0]
t1_loss_steep = 0.5 * t1_h_steep * (t1_grid - t1_star) ** 2  # -> [45.0, 20.0, 5.0, 0.0, 5.0, 20.0]
print("steep losses:", np.round(t1_loss_steep, 3).tolist())  # -> [45.0, 20.0, 5.0, 0.0, 5.0, 20.0]
t1_position = 3.0  # -> 3.0
print("test position:", t1_position)  # -> 3.0
t1_grad_flat = t1_h_flat * (t1_position - t1_star)  # -> 2.0
print("flat gradient:", round(float(t1_grad_flat), 3))  # -> 2.0
t1_grad_steep = t1_h_steep * (t1_position - t1_star)  # -> 20.0
print("steep gradient:", round(float(t1_grad_steep), 3))  # -> 20.0
assert round(float(t1_grad_flat), 3) == 2.0
assert round(float(t1_grad_steep), 3) == 20.0

plt.figure(figsize=(4.8, 2.8))
plt.plot(t1_grid, t1_loss_flat, marker="o", label="h=1", color="gray")
plt.plot(t1_grid, t1_loss_steep, marker="s", label="h=10", color="crimson")
plt.xlabel("w")
plt.ylabel("loss")
plt.title("Toy 1 · curvature changes slope")
plt.legend()
plt.show()

▶ What you'll see: at `w=3`, steep curvature makes gradient `20` instead of `2`.

### ✍️ Toy 2 · Newton divides by curvature

On a quadratic, `gradient / curvature` is exactly the distance back to the optimum.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t2_w0 = 4.0  # -> 4.0
print("start w:", t2_w0)  # -> 4.0
t2_star = 1.5  # -> 1.5
print("optimum:", t2_star)  # -> 1.5
t2_h = 8.0  # -> 8.0
print("curvature:", t2_h)  # -> 8.0
t2_grad = t2_h * (t2_w0 - t2_star)  # -> 20.0
print("gradient:", round(float(t2_grad), 3))  # -> 20.0
t2_newton_step = t2_grad / t2_h  # -> 2.5
print("Newton step length:", round(float(t2_newton_step), 3))  # -> 2.5
t2_w_newton = t2_w0 - t2_newton_step  # -> 1.5
print("Newton w:", round(float(t2_w_newton), 3))  # -> 1.5
t2_w_gd = t2_w0 - 0.05 * t2_grad  # -> 3.0
print("GD w:", round(float(t2_w_gd), 3))  # -> 3.0
t2_loss0 = 0.5 * t2_h * (t2_w0 - t2_star) ** 2  # -> 25.0
print("loss at start:", round(float(t2_loss0), 3))  # -> 25.0
t2_loss_gd = 0.5 * t2_h * (t2_w_gd - t2_star) ** 2  # -> 9.0
print("loss after GD:", round(float(t2_loss_gd), 3))  # -> 9.0
t2_loss_newton = 0.5 * t2_h * (t2_w_newton - t2_star) ** 2  # -> 0.0
print("loss after Newton:", round(float(t2_loss_newton), 3))  # -> 0.0
assert round(float(t2_w_newton), 3) == 1.5
assert round(float(t2_loss_newton), 3) == 0.0

t2_grid = np.linspace(1.0, 4.5, 8)  # -> [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]
print("plot grid:", np.round(t2_grid, 3).tolist())  # -> [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5]
t2_curve = 0.5 * t2_h * (t2_grid - t2_star) ** 2  # -> [1.0, 0.0, 1.0, 4.0, 9.0, 16.0, 25.0, 36.0]
print("loss curve:", np.round(t2_curve, 3).tolist())  # -> [1.0, 0.0, 1.0, 4.0, 9.0, 16.0, 25.0, 36.0]
plt.figure(figsize=(4.8, 2.8))
plt.plot(t2_grid, t2_curve, marker="o", color="purple")
plt.scatter([t2_w0, t2_w_gd, t2_w_newton], [t2_loss0, t2_loss_gd, t2_loss_newton], color=["red", "orange", "green"], zorder=3)
plt.xlabel("w")
plt.ylabel("loss")
plt.title("Toy 2 · Newton reaches the bottom")
plt.show()

▶ What you'll see: GD moves to `3.0`, but Newton lands exactly at the optimum `1.5`.

### ✍️ Toy 3 · A layer gradient is an outer product

For one example, each weight gradient is activation coordinate times output derivative coordinate.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t3_activation = np.array([1.0, -2.0, 0.5])  # -> [1.0, -2.0, 0.5]
print("activation:", t3_activation.tolist())  # -> [1.0, -2.0, 0.5]
t3_delta = np.array([0.4, -0.2])  # -> [0.4, -0.2]
print("backprop delta:", t3_delta.tolist())  # -> [0.4, -0.2]
t3_activation_col = t3_activation[:, None]  # -> shape (3, 1)
print("activation column shape:", t3_activation_col.shape)  # -> (3, 1)
t3_delta_row = t3_delta[None, :]  # -> shape (1, 2)
print("delta row shape:", t3_delta_row.shape)  # -> (1, 2)
t3_outer = t3_activation_col * t3_delta_row  # -> [[0.4, -0.2], [-0.8, 0.4], [0.2, -0.1]]
print("outer-product gradient:\n", np.round(t3_outer, 3))  # -> [[0.4,-0.2], [-0.8,0.4], [0.2,-0.1]]
assert t3_outer.shape == (3, 2)
assert np.allclose(np.round(t3_outer[1], 3), [-0.8, 0.4])

plt.figure(figsize=(4.2, 3.0))
plt.imshow(t3_outer, cmap="coolwarm")
plt.colorbar(label="gradient entry")
plt.xlabel("output coordinate")
plt.ylabel("input coordinate")
plt.title("Toy 3 · outer product")
plt.show()

▶ What you'll see: a 3-vector activation and a 2-vector delta make a `3×2` weight-gradient matrix.

### ✍️ Toy 4 · K-FAC factors expand with a Kronecker product

K-FAC stores two small covariance factors whose Kronecker product represents layer curvature.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t4_acts = np.array([[1.0, 0.0], [0.0, 2.0], [1.0, 1.0]])  # -> three 2D activations
print("activation batch:\n", t4_acts)  # -> [[1,0], [0,2], [1,1]]
t4_deltas = np.array([[0.5, -0.5], [1.0, 0.0], [0.0, 1.0]])  # -> three 2D deltas
print("delta batch:\n", t4_deltas)  # -> [[0.5,-0.5], [1,0], [0,1]]
t4_A = t4_acts.T @ t4_acts / t4_acts.shape[0]  # -> [[0.666667, 0.333333], [0.333333, 1.666667]]
print("A factor:\n", np.round(t4_A, 3))  # -> [[0.667,0.333], [0.333,1.667]]
t4_S = t4_deltas.T @ t4_deltas / t4_deltas.shape[0]  # -> [[0.416667, -0.083333], [-0.083333, 0.416667]]
print("S factor:\n", np.round(t4_S, 3))  # -> [[0.417,-0.083], [-0.083,0.417]]
t4_kron = np.kron(t4_S, t4_A)  # -> 4x4 Kronecker curvature
print("Kronecker curvature:\n", np.round(t4_kron, 3))  # -> [[0.278,0.139,-0.056,-0.028], [0.139,0.694,-0.028,-0.139], [-0.056,-0.028,0.278,0.139], [-0.028,-0.139,0.139,0.694]]
t4_factor_entries = t4_A.size + t4_S.size  # -> 8
print("factor entries:", t4_factor_entries)  # -> 8
t4_full_entries = t4_kron.size  # -> 16
print("expanded entries:", t4_full_entries)  # -> 16
assert t4_A.shape == (2, 2)
assert t4_kron.shape == (4, 4)

plt.figure(figsize=(4.2, 3.2))
plt.imshow(t4_kron, cmap="viridis")
plt.colorbar(label="curvature")
plt.title("Toy 4 · S ⊗ A")
plt.show()

▶ What you'll see: two `2×2` factors expand into one `4×4` curvature approximation.

### ✍️ Toy 5 · K-FAC preconditions on both sides

The K-FAC step left-multiplies by `A^{-1}` and right-multiplies by `S^{-1}`.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t5_A = np.array([[1.0, 0.25], [0.25, 2.0]])  # -> [[1.0, 0.25], [0.25, 2.0]]
print("A factor:\n", t5_A)  # -> [[1.0,0.25], [0.25,2.0]]
t5_S = np.array([[0.5, 0.1], [0.1, 1.0]])  # -> [[0.5, 0.1], [0.1, 1.0]]
print("S factor:\n", t5_S)  # -> [[0.5,0.1], [0.1,1.0]]
t5_G = np.array([[0.6, -0.2], [0.3, 0.4]])  # -> [[0.6, -0.2], [0.3, 0.4]]
print("raw gradient:\n", t5_G)  # -> [[0.6,-0.2], [0.3,0.4]]
t5_damp = 0.1  # -> 0.1
print("damping:", t5_damp)  # -> 0.1
t5_eta = 0.05  # -> 0.05
print("learning rate:", t5_eta)  # -> 0.05
t5_A_damped = t5_A + t5_damp * np.eye(2)  # -> [[1.1, 0.25], [0.25, 2.1]]
print("damped A:\n", np.round(t5_A_damped, 3))  # -> [[1.1,0.25], [0.25,2.1]]
t5_S_damped = t5_S + t5_damp * np.eye(2)  # -> [[0.6, 0.1], [0.1, 1.1]]
print("damped S:\n", np.round(t5_S_damped, 3))  # -> [[0.6,0.1], [0.1,1.1]]
t5_A_inv = np.linalg.inv(t5_A_damped)  # -> [[0.934, -0.111], [-0.111, 0.489]]
print("A inverse:\n", np.round(t5_A_inv, 3))  # -> [[0.934,-0.111], [-0.111,0.489]]
t5_S_inv = np.linalg.inv(t5_S_damped)  # -> [[1.692, -0.154], [-0.154, 0.923]]
print("S inverse:\n", np.round(t5_S_inv, 3))  # -> [[1.692,-0.154], [-0.154,0.923]]
t5_sgd_step = -t5_eta * t5_G  # -> [[-0.03, 0.01], [-0.015, -0.02]]
print("SGD step:\n", np.round(t5_sgd_step, 3))  # -> [[-0.03,0.01], [-0.015,-0.02]]
t5_kfac_step = -t5_eta * t5_A_inv @ t5_G @ t5_S_inv  # -> [[-0.046, 0.015], [-0.005, -0.009]]
print("K-FAC step:\n", np.round(t5_kfac_step, 3))  # -> [[-0.046,0.015], [-0.005,-0.009]]
t5_norm_sgd = np.linalg.norm(t5_sgd_step)  # -> 0.04031128874149275
print("SGD norm:", round(float(t5_norm_sgd), 4))  # -> 0.0403
t5_norm_kfac = np.linalg.norm(t5_kfac_step)  # -> 0.049846680209101484
print("K-FAC norm:", round(float(t5_norm_kfac), 4))  # -> 0.0498
assert t5_kfac_step.shape == t5_G.shape
assert round(float(t5_norm_kfac), 4) == 0.0498

plt.figure(figsize=(4.4, 2.8))
plt.bar(["SGD", "K-FAC"], [t5_norm_sgd, t5_norm_kfac], color=["gray", "seagreen"])
plt.ylabel("update norm")
plt.title("Toy 5 · two-sided preconditioning")
plt.show()

▶ What you'll see: the K-FAC update is not just the raw gradient scaled; it is rotated and rescaled by two inverse factors.

### ✍️ Toy 6 · Damping lifts tiny eigenvalues

Adding `λI` makes a singular covariance factor invertible by raising every eigenvalue.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)  # -> seeded Generator(PCG64)
t6_batch = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])  # -> columns are perfectly dependent
print("activation batch:\n", t6_batch)  # -> [[1,2], [2,4], [3,6]]
t6_A = t6_batch.T @ t6_batch / t6_batch.shape[0]  # -> [[4.666667, 9.333333], [9.333333, 18.666667]]
print("raw covariance:\n", np.round(t6_A, 3))  # -> [[4.667,9.333], [9.333,18.667]]
t6_eigs = np.linalg.eigvalsh(t6_A)  # -> [0.0, 23.333333]
print("raw eigenvalues:", np.round(t6_eigs, 6).tolist())  # -> [-0.0, 23.333333]
t6_damp = 0.1  # -> 0.1
print("damping:", t6_damp)  # -> 0.1
t6_A_damped = t6_A + t6_damp * np.eye(2)  # -> [[4.766667, 9.333333], [9.333333, 18.766667]]
print("damped covariance:\n", np.round(t6_A_damped, 3))  # -> [[4.767,9.333], [9.333,18.767]]
t6_eigs_damped = np.linalg.eigvalsh(t6_A_damped)  # -> [0.1, 23.433333]
print("damped eigenvalues:", np.round(t6_eigs_damped, 6).tolist())  # -> [0.1, 23.433333]
t6_condition = t6_eigs_damped[-1] / t6_eigs_damped[0]  # -> 234.33333333333323
print("damped condition number:", round(float(t6_condition), 3))  # -> 234.333
assert round(float(t6_eigs_damped[0]), 3) == 0.1
assert round(float(t6_condition), 3) == 234.333

plt.figure(figsize=(4.6, 2.8))
plt.bar(["raw smallest", "damped smallest"], [max(0.0, t6_eigs[0]), t6_eigs_damped[0]], color=["crimson", "seagreen"])
plt.ylabel("eigenvalue")
plt.title("Toy 6 · damping prevents zero divide")
plt.show()

▶ What you'll see: damping raises the smallest eigenvalue from `0` to `0.1`, making the inverse finite.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, matrix products, covariance factors, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for compact diagnostic plots.
np.random.seed(0) # make the examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — One affine signal and ReLU gate

**Goal.** Reproduce the lesson's two-input scratch pass, because K-FAC still begins with ordinary forward signals before it approximates curvature.

In [ ]:
x_b1 = np.array([1.5, -0.5]) # define the two visible input activations.
w_b1 = np.array([1.0, -0.1]) # define the two weights into one hidden unit.
b_b1 = 0.7 # define the bias from the lesson arithmetic.
z_b1 = float(x_b1 @ w_b1 + b_b1) # compute 1.0*1.5 + -0.1*(-0.5) + 0.7.
h_b1 = max(0.0, z_b1) # apply ReLU gating.
print("affine:", round(z_b1, 3), "ReLU:", round(h_b1, 3))
assert round(z_b1, 3) == 2.25
plt.figure(figsize=(4, 3)); plt.bar(["x0w0", "x1w1", "bias", "ReLU"], [x_b1[0]*w_b1[0], x_b1[1]*w_b1[1], b_b1, h_b1], color="teal")
plt.title("Basic 1: signal pieces"); plt.xticks(rotation=20); plt.show()

▶ What you'll see: the two weighted inputs plus bias sum to 2.25, and ReLU passes the positive value through.

👀 Takeaway: K-FAC preconditions training, but the signals it summarizes are the same activations produced by the forward pass.

### Basic 2 — A scalar gradient step

**Goal.** Compute the lesson's small first-order update, because second-order methods modify this familiar gradient-descent baseline.

In [ ]:
theta_b2 = 2.0 # current scalar parameter.
eta_b2 = 0.05 # learning rate from the lesson content.
g_b2 = 1.65 # ordinary gradient.
theta_new_b2 = theta_b2 - eta_b2 * g_b2 # gradient descent update.
print("new theta:", round(theta_new_b2, 3))
assert round(theta_new_b2, 3) == 1.917
plt.figure(figsize=(4, 3)); plt.bar(["before", "after"], [theta_b2, theta_new_b2], color=["gray", "seagreen"])
plt.title("Basic 2: one gradient nudge"); plt.ylabel("parameter value"); plt.show()

▶ What you'll see: the parameter moves from 2.000 to 1.917, a deliberately modest change.

👀 Takeaway: first-order training changes parameters by repeated small steps proportional to the raw gradient.

### Basic 3 — Softmax turns scores into comparisons

**Goal.** Convert the lesson score 2.25 and baseline 0.40 into a probability, because losses and gradients usually depend on comparisons, not absolute scores.

In [ ]:
scores_b3 = np.array([2.25, 0.40]) # model score and baseline score.
exp_b3 = np.exp(scores_b3) # exponentiate scores into positive weights.
prob_b3 = exp_b3[0] / np.sum(exp_b3) # normalize the first score's weight.
print("exp values:", np.round(exp_b3, 3))
print("probability:", round(float(prob_b3), 3))
assert round(float(prob_b3), 3) == 0.864
plt.figure(figsize=(4, 3)); plt.bar(["model", "baseline"], exp_b3, color=["purple", "gray"])
plt.title("Basic 3: exponentiated evidence"); plt.show()

▶ What you'll see: exp(2.25) is much larger than exp(0.40), so the normalized probability is about 0.864.

👀 Takeaway: optimization often follows probabilities whose scale depends sharply on the raw scores.

### Basic 4 — Normalize one signal

**Goal.** Compute the lesson's normalized value, because curvature and gradients are meaningful only relative to signal scale.

In [ ]:
value_b4 = 2.25 # signal value from the forward pass.
mean_b4 = 1.0 # running or batch mean.
var_b4 = 0.25 # running or batch variance.
eps_b4 = 1e-5 # numerical stabilizer.
norm_b4 = (value_b4 - mean_b4) / np.sqrt(var_b4 + eps_b4) # standardized signal.
print("normalized value:", round(float(norm_b4), 3))
assert round(float(norm_b4), 3) == 2.5
plt.figure(figsize=(4, 3)); plt.bar(["raw", "mean", "normalized"], [value_b4, mean_b4, norm_b4], color="orange")
plt.title("Basic 4: scale bookkeeping"); plt.show()

▶ What you'll see: the signal is 2.5 standard deviations above the mean.

👀 Takeaway: scale bookkeeping changes how large an update effectively feels.

### Basic 5 — Count activation memory

**Goal.** Reproduce the lesson's activation memory calculation, because second-order methods are constrained by hardware as much as algebra.

In [ ]:
batch_b5 = 4 # number of activation vectors.
width_b5 = 128 # length of each activation vector.
bytes_per_float_b5 = 4 # 32-bit float storage.
kb_b5 = batch_b5 * width_b5 * bytes_per_float_b5 / 1024 # convert bytes to KB.
print("activation memory KB:", round(kb_b5, 3))
assert round(kb_b5, 3) == 2.0
plt.figure(figsize=(4, 3)); plt.bar(["activations"], [kb_b5], color="steelblue")
plt.title("Basic 5: tiny activation block memory"); plt.ylabel("KB"); plt.show()

▶ What you'll see: even a tiny 4×128 activation block uses 2 KB in float32.

👀 Takeaway: practical optimization methods must respect the memory footprint of saved activations and curvature factors.

### Basic 6 — Build an activation covariance factor

**Goal.** Compute \(A=E[a^\top a]\), because K-FAC's left factor summarizes activation scale and correlation.

In [ ]:
acts_b6 = np.array([[1.5, -0.5], [0.5, 1.0], [1.0, 0.0], [-1.0, 0.5]]) # four activation rows.
A_b6 = acts_b6.T @ acts_b6 / acts_b6.shape[0] # average outer product.
print("A:\n", np.round(A_b6, 3))
assert np.allclose(np.round(A_b6, 3), [[1.125, -0.188], [-0.188, 0.375]])
plt.figure(figsize=(4, 3)); plt.imshow(A_b6, cmap="viridis"); plt.colorbar(label="covariance")
plt.title("Basic 6: activation factor A"); plt.show()

▶ What you'll see: the first activation coordinate has larger variance than the second.

👀 Takeaway: K-FAC learns how input coordinates are scaled before choosing a weight update.

### Basic 7 — Build a gradient covariance factor

**Goal.** Compute \(S=E[\delta^\top\delta]\), because K-FAC's right factor summarizes output-gradient scale and correlation.

In [ ]:
deltas_b7 = np.array([[1.65, -0.20], [0.50, 0.30], [1.00, -0.40], [-0.30, 0.20]]) # backpropagated derivatives.
S_b7 = deltas_b7.T @ deltas_b7 / deltas_b7.shape[0] # average derivative outer product.
print("S:\n", np.round(S_b7, 3))
assert np.allclose(np.round(S_b7, 3), [[1.016, -0.16], [-0.16, 0.082]])
plt.figure(figsize=(4, 3)); plt.imshow(S_b7, cmap="magma"); plt.colorbar(label="covariance")
plt.title("Basic 7: gradient factor S"); plt.show()

▶ What you'll see: the first output derivative has much larger second moment than the second.

👀 Takeaway: output-side gradient scale is a separate part of the layer's curvature geometry.

### Basic 8 — Compute a batch weight gradient

**Goal.** Average activation-derivative outer products, because the ordinary gradient \(G\) is the object K-FAC preconditions.

In [ ]:
acts_b8 = np.array([[1.5, -0.5], [0.5, 1.0], [1.0, 0.0], [-1.0, 0.5]]) # activation batch.
deltas_b8 = np.array([[1.65, -0.20], [0.50, 0.30], [1.00, -0.40], [-0.30, 0.20]]) # derivative batch.
G_b8 = acts_b8.T @ deltas_b8 / acts_b8.shape[0] # matrix gradient for a 2x2 layer.
print("G:\n", np.round(G_b8, 3))
assert np.allclose(np.round(G_b8, 3), [[1.006, -0.188], [-0.119, 0.125]])
plt.figure(figsize=(4, 3)); plt.imshow(G_b8, cmap="coolwarm"); plt.colorbar(label="gradient")
plt.title("Basic 8: ordinary layer gradient"); plt.show()

▶ What you'll see: a 2×2 gradient with positive and negative entries.

👀 Takeaway: K-FAC does not replace gradients; it rescales the gradient matrix using curvature estimates.

### Basic 9 — Add damping before inversion

**Goal.** Stabilize a covariance inverse, because tiny eigenvalues would make a preconditioned step explode.

In [ ]:
A_b9 = np.array([[1.125, -0.1875], [-0.1875, 0.375]]) # activation factor.
damping_b9 = 0.1 # diagonal damping strength.
A_damped_b9 = A_b9 + damping_b9 * np.eye(2) # lift every eigenvalue by damping.
eigs_b9 = np.linalg.eigvalsh(A_damped_b9) # inspect numerical stability.
print("damped eigenvalues:", np.round(eigs_b9, 3))
assert np.all(eigs_b9 > 0.1)
plt.figure(figsize=(4, 3)); plt.bar(["eig0", "eig1"], eigs_b9, color="seagreen")
plt.title("Basic 9: damped curvature eigenvalues"); plt.show()

▶ What you'll see: both damped eigenvalues are safely positive.

👀 Takeaway: damping limits how aggressively the inverse can amplify uncertain directions.

### Basic 10 — Apply the K-FAC formula once

**Goal.** Compute \(-\eta A^{-1}GS^{-1}\), because this is the lesson's central curvature-preconditioned update.

In [ ]:
A_b10 = np.array([[1.125, -0.1875], [-0.1875, 0.375]]) # activation factor.
S_b10 = np.array([[1.015625, -0.16], [-0.16, 0.0825]]) # gradient factor.
G_b10 = np.array([[1.00625, -0.1875], [-0.11875, 0.125]]) # ordinary gradient.
eta_b10 = 0.05 # learning rate.
damp_b10 = 0.1 # damping for both factors.
Delta_b10 = -eta_b10 * np.linalg.inv(A_b10 + damp_b10*np.eye(2)) @ G_b10 @ np.linalg.inv(S_b10 + damp_b10*np.eye(2))
print("K-FAC update:\n", np.round(Delta_b10, 3))
assert Delta_b10.shape == (2, 2)
plt.figure(figsize=(4, 3)); plt.imshow(Delta_b10, cmap="coolwarm"); plt.colorbar(label="update")
plt.title("Basic 10: preconditioned step"); plt.show()

▶ What you'll see: the update is a curvature-shaped matrix rather than a uniform multiple of `G_b10`.

👀 Takeaway: K-FAC's practical step is two small inverses wrapped around the ordinary gradient.

## 🟡 Easy

### Easy 1 — Compare SGD and K-FAC directions

**Goal.** Put first-order and K-FAC updates side by side, because preconditioning changes both magnitude and direction.

In [ ]:
A_e1 = np.array([[1.125, -0.1875], [-0.1875, 0.375]]) # activation covariance.
S_e1 = np.array([[1.015625, -0.16], [-0.16, 0.0825]]) # derivative covariance.
G_e1 = np.array([[1.00625, -0.1875], [-0.11875, 0.125]]) # ordinary gradient.
eta_e1, damp_e1 = 0.05, 0.1 # learning rate and damping.
sgd_e1 = -eta_e1 * G_e1 # first-order update.
kfac_e1 = -eta_e1 * np.linalg.inv(A_e1 + damp_e1*np.eye(2)) @ G_e1 @ np.linalg.inv(S_e1 + damp_e1*np.eye(2))
cos_e1 = float(np.sum(sgd_e1 * kfac_e1) / (np.linalg.norm(sgd_e1) * np.linalg.norm(kfac_e1)))
print("||SGD||:", round(float(np.linalg.norm(sgd_e1)), 4), "||K-FAC||:", round(float(np.linalg.norm(kfac_e1)), 4))
print("direction cosine:", round(cos_e1, 3))
assert round(float(np.linalg.norm(sgd_e1)), 4) == 0.0519
plt.figure(figsize=(4, 3)); plt.bar(["SGD", "K-FAC"], [np.linalg.norm(sgd_e1), np.linalg.norm(kfac_e1)], color=["gray", "green"])
plt.title("Easy 1: update norms"); plt.show()

▶ What you'll see: K-FAC is not merely `SGD` with a different learning rate; the direction cosine is below 1.

👀 Takeaway: preconditioning changes the geometry of the step, not just its scalar size.

### Easy 2 — Verify the Kronecker inverse identity

**Goal.** Check \((S\otimes A)^{-1}=S^{-1}\otimes A^{-1}\), because this identity is what makes K-FAC computationally attractive.

In [ ]:
A_e2 = np.array([[1.2, 0.2], [0.2, 0.7]]) # positive definite activation factor.
S_e2 = np.array([[0.9, -0.1], [-0.1, 0.4]]) # positive definite derivative factor.
left_e2 = np.linalg.inv(np.kron(S_e2, A_e2)) # inverse of the full Kronecker matrix.
right_e2 = np.kron(np.linalg.inv(S_e2), np.linalg.inv(A_e2)) # Kronecker of small inverses.
err_e2 = float(np.max(np.abs(left_e2 - right_e2))) # maximum numerical difference.
print("max identity error:", err_e2)
assert err_e2 < 1e-12
plt.figure(figsize=(4, 3)); plt.imshow(left_e2 - right_e2, cmap="coolwarm"); plt.colorbar(label="difference")
plt.title("Easy 2: inverse identity error"); plt.show()

▶ What you'll see: the difference heatmap is essentially zero everywhere.

👀 Takeaway: two small inverses can represent the action of one much larger structured inverse.

### Easy 3 — Show damping shrinks an aggressive step

**Goal.** Sweep damping values, because larger damping moves the method away from pure Newton and toward safer, smaller updates.

In [ ]:
A_e3 = np.array([[1.125, -0.1875], [-0.1875, 0.375]])
S_e3 = np.array([[1.015625, -0.16], [-0.16, 0.0825]])
G_e3 = np.array([[1.00625, -0.1875], [-0.11875, 0.125]])
damps_e3 = np.array([0.01, 0.05, 0.1, 0.5, 1.0]) # stability strengths to compare.
norms_e3 = []
for d_e3 in damps_e3:
    step_e3 = -0.05 * np.linalg.inv(A_e3 + d_e3*np.eye(2)) @ G_e3 @ np.linalg.inv(S_e3 + d_e3*np.eye(2))
    norms_e3.append(float(np.linalg.norm(step_e3)))
print("step norms:", np.round(norms_e3, 4))
assert norms_e3[0] > norms_e3[-1]
plt.figure(figsize=(4.6, 3)); plt.plot(damps_e3, norms_e3, marker="o", color="crimson")
plt.title("Easy 3: damping vs update size"); plt.xlabel("damping"); plt.ylabel("step norm"); plt.show()

▶ What you'll see: the update norm falls as damping increases.

👀 Takeaway: damping is a safety knob that prevents noisy curvature estimates from taking oversized steps.

### Easy 4 — Estimate factors from two minibatches

**Goal.** Compare covariance estimates across minibatches, because K-FAC factors are statistical estimates rather than exact constants.

In [ ]:
rng_e4 = np.random.default_rng(4) # reproducible minibatches.
acts1_e4 = rng_e4.normal(loc=0.0, scale=1.0, size=(8, 2)) # first activation minibatch.
acts2_e4 = rng_e4.normal(loc=0.0, scale=1.0, size=(8, 2)) # second activation minibatch.
A1_e4 = acts1_e4.T @ acts1_e4 / acts1_e4.shape[0] # first factor estimate.
A2_e4 = acts2_e4.T @ acts2_e4 / acts2_e4.shape[0] # second factor estimate.
diff_e4 = float(np.linalg.norm(A1_e4 - A2_e4))
print("A1:\n", np.round(A1_e4, 3))
print("A2:\n", np.round(A2_e4, 3))
print("factor difference norm:", round(diff_e4, 3))
assert diff_e4 > 0.1
plt.figure(figsize=(4, 3)); plt.bar(["batch1 trace", "batch2 trace"], [np.trace(A1_e4), np.trace(A2_e4)], color="purple")
plt.title("Easy 4: minibatch factor variability"); plt.show()

▶ What you'll see: the two minibatches produce similar-shaped but not identical covariance factors.

👀 Takeaway: K-FAC must balance curvature information with estimator noise from minibatches.

### Easy 5 — Translate factor storage into memory savings

**Goal.** Count entries for full curvature versus K-FAC factors, because the approximation exists to make second-order information fit in memory.

In [ ]:
sizes_e5 = np.array([16, 64, 256, 1024]) # square layer widths to compare.
full_entries_e5 = (sizes_e5 * sizes_e5) ** 2 # full curvature entries for W flattened.
factor_entries_e5 = sizes_e5 ** 2 + sizes_e5 ** 2 # A plus S entries.
ratios_e5 = full_entries_e5 / factor_entries_e5
print("ratios:", ratios_e5.astype(int))
assert int(ratios_e5[-1]) == 524288
plt.figure(figsize=(5, 3)); plt.plot(sizes_e5, ratios_e5, marker="o", color="teal")
plt.title("Easy 5: K-FAC storage ratio"); plt.xlabel("layer width"); plt.ylabel("full entries / factor entries"); plt.yscale("log"); plt.show()

▶ What you'll see: the storage advantage grows rapidly with layer width.

👀 Takeaway: K-FAC keeps second-order training practical by replacing one giant matrix with two layer-sized factors.

## 🔴 Advanced

### Advanced 1 — Match matrix preconditioning to full Kronecker preconditioning

**Goal.** Verify that the matrix formula \(A^{-1}GS^{-1}\) matches applying \((S\otimes A)^{-1}\) to a flattened gradient when column-major vectorization is used.

In [ ]:
A_a1 = np.array([[1.2, 0.2], [0.2, 0.7]])
S_a1 = np.array([[0.9, -0.1], [-0.1, 0.4]])
G_a1 = np.array([[1.0, -0.3], [0.2, 0.5]])
mat_step_a1 = np.linalg.inv(A_a1) @ G_a1 @ np.linalg.inv(S_a1) # two-sided K-FAC action.
vecG_a1 = G_a1.reshape(-1, order="F") # column-major vectorization.
full_step_a1 = np.linalg.inv(np.kron(S_a1, A_a1)) @ vecG_a1 # full Kronecker inverse action.
max_err_a1 = float(np.max(np.abs(mat_step_a1.reshape(-1, order="F") - full_step_a1)))
print("max equivalence error:", max_err_a1)
assert max_err_a1 < 1e-12
plt.figure(figsize=(4, 3)); plt.imshow(mat_step_a1, cmap="coolwarm"); plt.colorbar(label="preconditioned gradient")
plt.title("Advanced 1: matrix-form preconditioner"); plt.show()

▶ What you'll see: the numerical error is near machine precision, confirming both views are the same operation.

👀 Takeaway: K-FAC is not hand-wavy matrix decoration; it is the structured full-curvature operation written efficiently.

### Advanced 2 — Train a tiny linear model with SGD versus K-FAC

**Goal.** Fit a noisy linear regression with both updates, because curvature-aware scaling should reduce loss quickly when factor estimates are stable.

In [ ]:
rng_a2 = np.random.default_rng(2)
X_a2 = rng_a2.normal(size=(40, 2)) # design matrix / activations.
true_W_a2 = np.array([[1.5], [-2.0]]) # target linear weights.
y_a2 = X_a2 @ true_W_a2 + 0.05 * rng_a2.normal(size=(40, 1)) # noisy targets.
W_sgd_a2 = np.zeros((2, 1)); W_kfac_a2 = np.zeros((2, 1)) # two models from same start.
loss_sgd_a2 = []; loss_kfac_a2 = []
for epoch_a2 in range(30):
    err_sgd_a2 = X_a2 @ W_sgd_a2 - y_a2
    G_sgd_a2 = X_a2.T @ err_sgd_a2 / X_a2.shape[0]
    W_sgd_a2 -= 0.2 * G_sgd_a2
    err_kfac_a2 = X_a2 @ W_kfac_a2 - y_a2
    G_kfac_a2 = X_a2.T @ err_kfac_a2 / X_a2.shape[0]
    A_kfac_a2 = X_a2.T @ X_a2 / X_a2.shape[0] + 0.05 * np.eye(2)
    W_kfac_a2 -= 0.8 * np.linalg.inv(A_kfac_a2) @ G_kfac_a2
    loss_sgd_a2.append(float(np.mean((X_a2 @ W_sgd_a2 - y_a2) ** 2)))
    loss_kfac_a2.append(float(np.mean((X_a2 @ W_kfac_a2 - y_a2) ** 2)))
print("final losses SGD/K-FAC:", round(loss_sgd_a2[-1], 5), round(loss_kfac_a2[-1], 5))
assert loss_kfac_a2[-1] < loss_sgd_a2[-1]
plt.figure(figsize=(5, 3)); plt.plot(loss_sgd_a2, label="SGD"); plt.plot(loss_kfac_a2, label="K-FAC-like")
plt.title("Advanced 2: curvature-aware convergence"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.show()

▶ What you'll see: the K-FAC-like curve drops faster and ends below the SGD curve for this scaled linear problem.

👀 Takeaway: when the curvature approximation is accurate, preconditioning can convert slow zig-zagging into direct progress.

### Advanced 3 — Show why scale imbalance hurts SGD

**Goal.** Rescale one input feature by 20×, because first-order updates become ill-conditioned when coordinates have very different curvature.

In [ ]:
rng_a3 = np.random.default_rng(3)
X_a3 = rng_a3.normal(size=(60, 2)); X_a3[:, 0] *= 20.0 # one high-scale feature.
true_W_a3 = np.array([[0.1], [2.0]])
y_a3 = X_a3 @ true_W_a3
A_a3 = X_a3.T @ X_a3 / X_a3.shape[0]
eigs_a3 = np.linalg.eigvalsh(A_a3)
condition_a3 = float(eigs_a3[-1] / eigs_a3[0])
print("feature curvature eigenvalues:", np.round(eigs_a3, 3))
print("condition number:", round(condition_a3, 1))
assert condition_a3 > 100
plt.figure(figsize=(4, 3)); plt.bar(["small eig", "large eig"], eigs_a3, color=["orange", "crimson"])
plt.title("Advanced 3: scale imbalance curvature"); plt.yscale("log"); plt.show()

▶ What you'll see: the large-scale feature creates a much larger curvature eigenvalue.

👀 Takeaway: K-FAC's activation factor directly targets the scale imbalance that forces SGD to use tiny steps.

### Advanced 4 — Track eigenvalues before and after damping

**Goal.** Inspect eigenvalues across a range of damping values, because the inverse preconditioner is only safe when its smallest eigenvalues are controlled.

In [ ]:
A_a4 = np.array([[2.0, 1.99], [1.99, 1.9801]]) # nearly rank-one covariance.
damps_a4 = np.array([0.0, 1e-3, 1e-2, 1e-1, 1.0])
smallest_a4 = []
conds_a4 = []
for d_a4 in damps_a4:
    eigs_a4 = np.linalg.eigvalsh(A_a4 + d_a4 * np.eye(2))
    smallest_a4.append(float(eigs_a4[0]))
    conds_a4.append(float(eigs_a4[-1] / eigs_a4[0]))
print("smallest eigenvalues:", np.round(smallest_a4, 6))
print("conditions:", np.round(conds_a4, 1))
assert conds_a4[-1] < conds_a4[1]
plt.figure(figsize=(5, 3)); plt.plot(damps_a4, conds_a4, marker="o", color="navy")
plt.title("Advanced 4: damping improves conditioning"); plt.xlabel("damping"); plt.ylabel("condition number"); plt.yscale("log"); plt.show()

▶ What you'll see: as damping grows, the condition number falls by orders of magnitude.

👀 Takeaway: damping is the practical bridge between a mathematically attractive inverse and a numerically safe update.

### Advanced 5 — Compare full curvature memory with factor memory in MB

**Goal.** Convert entry counts into megabytes, because the full second-order matrix becomes impossible before the algebra becomes confusing.

In [ ]:
widths_a5 = np.array([64, 128, 256, 512]) # square layer widths.
float_bytes_a5 = 4 # float32 storage.
full_mb_a5 = ((widths_a5 * widths_a5) ** 2) * float_bytes_a5 / (1024 ** 2) # full curvature memory.
factor_mb_a5 = (2 * widths_a5 ** 2) * float_bytes_a5 / (1024 ** 2) # A and S memory.
print("full MB:", np.round(full_mb_a5, 1))
print("factor MB:", np.round(factor_mb_a5, 3))
assert round(float(full_mb_a5[-1]), 1) == 262144.0
plt.figure(figsize=(5, 3)); plt.plot(widths_a5, full_mb_a5, marker="o", label="full curvature")
plt.plot(widths_a5, factor_mb_a5, marker="o", label="K-FAC factors")
plt.yscale("log"); plt.title("Advanced 5: memory wall"); plt.xlabel("layer width"); plt.ylabel("MB, log scale"); plt.legend(); plt.show()

▶ What you'll see: at width 512, full curvature is hundreds of gigabytes while factors are only a few megabytes.

👀 Takeaway: K-FAC is a second-order compromise designed for the memory scale of neural-network layers.